# Silver configuration mappings

## Purpose

Create the five small, versioned configuration tables that Silver transformations will use. These tables separate approved business decisions from transformation code. This milestone does not clean Bronze, build the remaining Silver outputs, or create Gold tables.

## Safe reruns

`CREATE TABLE IF NOT EXISTS` preserves reviewed rows. Seeded rules (MGB codes and DPWH statuses) use a deterministic `MERGE`, so rerunning the notebook does not append duplicates. Coverage queries read Bronze but never write to it.

## 1. Place-name aliases

Grain: one context-specific source alias per alias version. Region, province, municipality, and place type are part of the natural key so a common name such as `Poblacion` is never treated as a global match. The table starts empty because no alias has received human approval.

In [ ]:
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `02-silver`;

CREATE TABLE IF NOT EXISTS `02-silver`.config_place_name_alias (
    source_system STRING NOT NULL,
    raw_place_name STRING NOT NULL,
    raw_region_name STRING NOT NULL,
    raw_province_name STRING NOT NULL,
    raw_city_municipality_name STRING NOT NULL,
    place_type STRING NOT NULL,
    canonical_psgc_code STRING,
    canonical_place_name STRING,
    match_rule STRING,
    alias_version STRING NOT NULL,
    psgc_version STRING,
    approval_status STRING NOT NULL,
    approved_by STRING,
    approved_at TIMESTAMP,
    is_active BOOLEAN NOT NULL,
    source_reference STRING,
    notes STRING
)
USING DELTA
COMMENT 'Context-specific, reviewed source place aliases for Silver matching';

In [ ]:
SELECT
    COUNT(*) AS row_count,
    COUNT_IF(approval_status = 'APPROVED' AND is_active) AS active_approved_rows,
    COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
        'source_system', source_system,
        'raw_place_name', raw_place_name,
        'raw_region_name', raw_region_name,
        'raw_province_name', raw_province_name,
        'raw_city_municipality_name', raw_city_municipality_name,
        'place_type', place_type,
        'alias_version', alias_version
    )) AS duplicate_natural_keys
FROM `02-silver`.config_place_name_alias;

## 2. Project-category mappings

Grain: one source category and infrastructure-type pair per taxonomy version. At least one raw field must be populated. The table starts empty because observed raw categories have not yet been approved into a project taxonomy.

In [ ]:
CREATE TABLE IF NOT EXISTS `02-silver`.config_project_category_mapping (
    source_system STRING NOT NULL,
    raw_category STRING NOT NULL,
    source_infra_type STRING NOT NULL,
    standardized_sector STRING,
    standardized_category STRING,
    mapping_rule STRING,
    taxonomy_version STRING NOT NULL,
    approval_status STRING NOT NULL,
    approved_by STRING,
    approved_at TIMESTAMP,
    is_active BOOLEAN NOT NULL,
    source_reference STRING,
    notes STRING
)
USING DELTA
COMMENT 'Reviewed project category and infrastructure type mappings by taxonomy version';

In [ ]:
SELECT
    COUNT(*) AS row_count,
    COUNT_IF(approval_status = 'APPROVED' AND is_active) AS active_approved_rows,
    COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
        'source_system', source_system,
        'raw_category', raw_category,
        'source_infra_type', source_infra_type,
        'taxonomy_version', taxonomy_version
    )) AS duplicate_natural_keys
FROM `02-silver`.config_project_category_mapping;

## 3. Project-status mappings

Grain: one source status per mapping version. Version `2026-10-v1` seeds the five legitimate DPWH source statuses approved in GitHub issue #84. `For Procurement` maps to `Ongoing` and `ACTIVE`. `Not Yet Started` and `Terminated` map to `Inactive` and `INACTIVE`.

Blank, shifted, or malformed statuses, such as numbers or place names, get no rule. They stay `UNMAPPED` with `NULL` standardized values and remain visible in coverage output.

`Delayed` is not a supported target because the sources do not provide a reliable target completion date. Long-running and stalled are future derived measures, not status mappings.

In [ ]:
CREATE TABLE IF NOT EXISTS `02-silver`.config_project_status_mapping (
    source_system STRING NOT NULL,
    source_status STRING NOT NULL,
    standardized_status STRING,
    status_group STRING,
    mapping_rule STRING,
    status_mapping_version STRING NOT NULL,
    approval_status STRING NOT NULL,
    approved_by STRING,
    approved_at TIMESTAMP,
    is_active BOOLEAN NOT NULL,
    source_reference STRING,
    notes STRING
)
USING DELTA
COMMENT 'Reviewed source-to-standard project status mappings';

MERGE INTO `02-silver`.config_project_status_mapping AS target
USING (
    SELECT *
    FROM VALUES
        ('dpwh_projects', 'Completed', 'Completed', 'FINISHED', 'Approved DPWH status mapping v1', '2026-10-v1', 'APPROVED', TRUE, 'GitHub issue #84: DPWH project status mapping v1'),
        ('dpwh_projects', 'On-Going', 'Ongoing', 'ACTIVE', 'Approved DPWH status mapping v1', '2026-10-v1', 'APPROVED', TRUE, 'GitHub issue #84: DPWH project status mapping v1'),
        ('dpwh_projects', 'For Procurement', 'Ongoing', 'ACTIVE', 'Approved DPWH status mapping v1', '2026-10-v1', 'APPROVED', TRUE, 'GitHub issue #84: DPWH project status mapping v1'),
        ('dpwh_projects', 'Not Yet Started', 'Inactive', 'INACTIVE', 'Approved DPWH status mapping v1', '2026-10-v1', 'APPROVED', TRUE, 'GitHub issue #84: DPWH project status mapping v1'),
        ('dpwh_projects', 'Terminated', 'Inactive', 'INACTIVE', 'Approved DPWH status mapping v1', '2026-10-v1', 'APPROVED', TRUE, 'GitHub issue #84: DPWH project status mapping v1')
    AS seed(
        source_system, source_status, standardized_status, status_group,
        mapping_rule, status_mapping_version, approval_status, is_active,
        source_reference
    )
) AS source
ON target.source_system = source.source_system
AND target.source_status = source.source_status
AND target.status_mapping_version = source.status_mapping_version
WHEN MATCHED AND NOT (
    target.standardized_status <=> source.standardized_status
    AND target.status_group <=> source.status_group
    AND target.mapping_rule <=> source.mapping_rule
    AND target.approval_status <=> source.approval_status
    AND target.is_active <=> source.is_active
    AND target.source_reference <=> source.source_reference
) THEN UPDATE SET
    standardized_status = source.standardized_status,
    status_group = source.status_group,
    mapping_rule = source.mapping_rule,
    approval_status = source.approval_status,
    is_active = source.is_active,
    source_reference = source.source_reference
WHEN NOT MATCHED THEN INSERT (
    source_system, source_status, standardized_status, status_group,
    mapping_rule, status_mapping_version, approval_status, is_active,
    source_reference
) VALUES (
    source.source_system, source.source_status, source.standardized_status,
    source.status_group, source.mapping_rule, source.status_mapping_version,
    source.approval_status, source.is_active, source.source_reference
);

In [ ]:
SELECT
    COUNT(*) AS row_count,
    COUNT_IF(approval_status = 'APPROVED' AND is_active) AS active_approved_rows,
    COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
        'source_system', source_system,
        'source_status', source_status,
        'status_mapping_version', status_mapping_version
    )) AS duplicate_natural_keys
FROM `02-silver`.config_project_status_mapping;

## 4. MGB susceptibility mappings

Grain: one raw MGB code per source-contract and mapping version. Only the four documented codes are seeded. Blank values and `No rating` remain unmapped and visible in coverage output until a human decision is recorded.

In [ ]:
CREATE TABLE IF NOT EXISTS `02-silver`.config_mgb_susceptibility_mapping (
    source_system STRING NOT NULL,
    raw_susceptibility_code STRING NOT NULL,
    standardized_level STRING,
    severity_rank INT,
    mapping_rule STRING,
    source_version STRING NOT NULL,
    mapping_version STRING NOT NULL,
    approval_status STRING NOT NULL,
    approved_by STRING,
    approved_at TIMESTAMP,
    is_active BOOLEAN NOT NULL,
    source_reference STRING,
    notes STRING
)
USING DELTA
COMMENT 'Versioned DENR MGB flood susceptibility code mappings';

MERGE INTO `02-silver`.config_mgb_susceptibility_mapping AS target
USING (
    SELECT *
    FROM VALUES
        ('DENR MGB flood susceptibility', 'LF', 'Low', 1, 'Official code-to-label contract', 'unversioned-official-code-contract', '2026-10-v1', 'APPROVED', TRUE, 'DENR MGB official susceptibility codes'),
        ('DENR MGB flood susceptibility', 'MF', 'Moderate', 2, 'Official code-to-label contract', 'unversioned-official-code-contract', '2026-10-v1', 'APPROVED', TRUE, 'DENR MGB official susceptibility codes'),
        ('DENR MGB flood susceptibility', 'HF', 'High', 3, 'Official code-to-label contract', 'unversioned-official-code-contract', '2026-10-v1', 'APPROVED', TRUE, 'DENR MGB official susceptibility codes'),
        ('DENR MGB flood susceptibility', 'VHF', 'Very High', 4, 'Official code-to-label contract', 'unversioned-official-code-contract', '2026-10-v1', 'APPROVED', TRUE, 'DENR MGB official susceptibility codes')
    AS seed(
        source_system, raw_susceptibility_code, standardized_level, severity_rank,
        mapping_rule, source_version, mapping_version, approval_status, is_active,
        source_reference
    )
) AS source
ON target.source_system = source.source_system
AND target.raw_susceptibility_code = source.raw_susceptibility_code
AND target.source_version = source.source_version
AND target.mapping_version = source.mapping_version
WHEN MATCHED AND NOT (
    target.standardized_level <=> source.standardized_level
    AND target.severity_rank <=> source.severity_rank
    AND target.mapping_rule <=> source.mapping_rule
    AND target.approval_status <=> source.approval_status
    AND target.is_active <=> source.is_active
    AND target.source_reference <=> source.source_reference
) THEN UPDATE SET
    standardized_level = source.standardized_level,
    severity_rank = source.severity_rank,
    mapping_rule = source.mapping_rule,
    approval_status = source.approval_status,
    is_active = source.is_active,
    source_reference = source.source_reference
WHEN NOT MATCHED THEN INSERT (
    source_system, raw_susceptibility_code, standardized_level, severity_rank,
    mapping_rule, source_version, mapping_version, approval_status, is_active,
    source_reference
) VALUES (
    source.source_system, source.raw_susceptibility_code, source.standardized_level,
    source.severity_rank, source.mapping_rule, source.source_version,
    source.mapping_version, source.approval_status, source.is_active,
    source.source_reference
);

In [ ]:
SELECT
    raw_susceptibility_code,
    standardized_level,
    severity_rank,
    source_version,
    mapping_version,
    approval_status,
    is_active
FROM `02-silver`.config_mgb_susceptibility_mapping
ORDER BY severity_rank;

## 5. Manual geographic matches

Grain: one explicit source-record override per mapping version. This table is an exception path, not the default matching method. It starts empty and may remain empty. Every future row must identify its source record, approved target PSGC code, reason, and approver.

In [ ]:
CREATE TABLE IF NOT EXISTS `02-silver`.config_manual_geographic_match (
    source_system STRING NOT NULL,
    record_type STRING NOT NULL,
    source_record_id STRING NOT NULL,
    raw_place_name STRING,
    raw_region_name STRING,
    raw_province_name STRING,
    raw_city_municipality_name STRING,
    canonical_psgc_code STRING,
    canonical_place_name STRING,
    override_reason STRING,
    mapping_version STRING NOT NULL,
    psgc_version STRING,
    approval_status STRING NOT NULL,
    approved_by STRING,
    approved_at TIMESTAMP,
    is_active BOOLEAN NOT NULL,
    source_reference STRING,
    notes STRING
)
USING DELTA
COMMENT 'Reviewed source-record-specific geographic exceptions';

In [ ]:
SELECT
    COUNT(*) AS row_count,
    COUNT_IF(approval_status = 'APPROVED' AND is_active) AS active_approved_rows,
    COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
        'source_system', source_system,
        'record_type', record_type,
        'source_record_id', source_record_id,
        'mapping_version', mapping_version
    )) AS duplicate_natural_keys
FROM `02-silver`.config_manual_geographic_match;

## Mapping coverage

The following query profiles only the distinct mapping domains needed for review. It does not transform or rewrite Bronze. The MGB table is grouped once in this notebook; the 2 GB source is not cached, sorted, or rescanned per code. Pending and unmapped values remain visible instead of being guessed.

In [ ]:
WITH raw_project_category AS (
    SELECT
        'dpwh_projects' AS source_system,
        COALESCE(TRIM(category), '') AS raw_category,
        '' AS source_infra_type,
        COUNT(*) AS source_rows
    FROM `01-bronze`.dpwh_projects
    GROUP BY COALESCE(TRIM(category), '')
    UNION ALL
    SELECT
        'flood_control_projects' AS source_system,
        COALESCE(TRIM(type_of_work), '') AS raw_category,
        COALESCE(TRIM(infra_type), '') AS source_infra_type,
        COUNT(*) AS source_rows
    FROM `01-bronze`.flood_control_projects
    GROUP BY
        COALESCE(TRIM(type_of_work), ''),
        COALESCE(TRIM(infra_type), '')
),
raw_project_status AS (
    SELECT
        'dpwh_projects' AS source_system,
        COALESCE(TRIM(status), '') AS source_status,
        COUNT(*) AS source_rows
    FROM `01-bronze`.dpwh_projects
    GROUP BY COALESCE(TRIM(status), '')
),
raw_place_alias AS (
    SELECT
        'dpwh_projects' AS source_system,
        COALESCE(TRIM(reported_region), '') AS raw_place_name,
        COUNT(*) AS source_rows
    FROM `01-bronze`.dpwh_projects
    GROUP BY COALESCE(TRIM(reported_region), '')
    UNION ALL
    SELECT
        'flood_control_projects' AS source_system,
        COALESCE(TRIM(reported_region), '') AS raw_place_name,
        COUNT(*) AS source_rows
    FROM `01-bronze`.flood_control_projects
    GROUP BY COALESCE(TRIM(reported_region), '')
),
raw_mgb AS (
    SELECT
        COALESCE(UPPER(TRIM(flood_susceptibility_code)), '') AS raw_code,
        COUNT(*) AS source_rows
    FROM `01-bronze`.flood_susceptibility
    GROUP BY COALESCE(UPPER(TRIM(flood_susceptibility_code)), '')
),
coverage AS (
    SELECT
        'project_category' AS mapping_domain,
        raw.source_system,
        CONCAT(raw.raw_category, ' | ', raw.source_infra_type) AS raw_value,
        raw.source_rows,
        mapping.approval_status,
        mapping.taxonomy_version AS mapping_version,
        mapping.is_active
    FROM raw_project_category AS raw
    LEFT JOIN `02-silver`.config_project_category_mapping AS mapping
        ON raw.source_system = mapping.source_system
        AND raw.raw_category = mapping.raw_category
        AND raw.source_infra_type = mapping.source_infra_type
        AND mapping.is_active
    UNION ALL
    SELECT
        'project_status' AS mapping_domain,
        raw.source_system,
        raw.source_status AS raw_value,
        raw.source_rows,
        mapping.approval_status,
        mapping.status_mapping_version AS mapping_version,
        mapping.is_active
    FROM raw_project_status AS raw
    LEFT JOIN `02-silver`.config_project_status_mapping AS mapping
        ON raw.source_system = mapping.source_system
        AND raw.source_status = mapping.source_status
        AND mapping.is_active
    UNION ALL
    SELECT
        'place_alias' AS mapping_domain,
        raw.source_system,
        raw.raw_place_name AS raw_value,
        raw.source_rows,
        mapping.approval_status,
        mapping.alias_version AS mapping_version,
        mapping.is_active
    FROM raw_place_alias AS raw
    LEFT JOIN `02-silver`.config_place_name_alias AS mapping
        ON raw.source_system = mapping.source_system
        AND raw.raw_place_name = mapping.raw_place_name
        AND mapping.raw_region_name = ''
        AND mapping.raw_province_name = ''
        AND mapping.raw_city_municipality_name = ''
        AND mapping.place_type = 'region'
        AND mapping.is_active
    UNION ALL
    SELECT
        'mgb_susceptibility' AS mapping_domain,
        'flood_susceptibility' AS source_system,
        raw.raw_code AS raw_value,
        raw.source_rows,
        mapping.approval_status,
        mapping.mapping_version,
        mapping.is_active
    FROM raw_mgb AS raw
    LEFT JOIN `02-silver`.config_mgb_susceptibility_mapping AS mapping
        ON raw.raw_code = mapping.raw_susceptibility_code
        AND mapping.is_active
)
SELECT
    mapping_domain,
    source_system,
    raw_value,
    source_rows,
    COALESCE(approval_status, 'UNMAPPED') AS review_state,
    mapping_version,
    CASE
        WHEN approval_status = 'APPROVED' AND is_active THEN 'READY'
        ELSE 'REVIEW_REQUIRED'
    END AS coverage_status
FROM coverage
ORDER BY mapping_domain, source_rows DESC, raw_value;

## Final summary

The next query reports table size, active approved rules, pending review rows, and visible mapping version. Empty tables are intentional until reviewers approve source-specific rules. Run the Silver configuration validator after this notebook.

In [ ]:
SELECT 'config_place_name_alias' AS table_name, COUNT(*) AS row_count,
    COUNT_IF(approval_status = 'APPROVED' AND is_active) AS active_approved_rows,
    COUNT_IF(approval_status = 'PENDING_REVIEW') AS pending_rows,
    COALESCE(MAX(alias_version), 'NO_RULES_YET') AS current_version
FROM `02-silver`.config_place_name_alias
UNION ALL
SELECT 'config_project_category_mapping', COUNT(*),
    COUNT_IF(approval_status = 'APPROVED' AND is_active),
    COUNT_IF(approval_status = 'PENDING_REVIEW'),
    COALESCE(MAX(taxonomy_version), 'NO_RULES_YET')
FROM `02-silver`.config_project_category_mapping
UNION ALL
SELECT 'config_project_status_mapping', COUNT(*),
    COUNT_IF(approval_status = 'APPROVED' AND is_active),
    COUNT_IF(approval_status = 'PENDING_REVIEW'),
    COALESCE(MAX(status_mapping_version), 'NO_RULES_YET')
FROM `02-silver`.config_project_status_mapping
UNION ALL
SELECT 'config_mgb_susceptibility_mapping', COUNT(*),
    COUNT_IF(approval_status = 'APPROVED' AND is_active),
    COUNT_IF(approval_status = 'PENDING_REVIEW'),
    COALESCE(MAX(mapping_version), 'NO_RULES_YET')
FROM `02-silver`.config_mgb_susceptibility_mapping
UNION ALL
SELECT 'config_manual_geographic_match', COUNT(*),
    COUNT_IF(approval_status = 'APPROVED' AND is_active),
    COUNT_IF(approval_status = 'PENDING_REVIEW'),
    COALESCE(MAX(mapping_version), 'NO_RULES_YET')
FROM `02-silver`.config_manual_geographic_match
ORDER BY table_name;